In [1]:
# Importar librerías
import pandas as pd
from src.config import cols_resultados, cols_balance, cols_cashflow
from src.funcionesExtract import *

In [ ]:
# Obtener lista de tickers del S&P 500 desde el fichero constituents.csv
# Si no existe el fichero, lo descarga desde GitHub
# Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
ruta_archivo = descargar_constituents(force_update=False) 

# Cargar datos
df_tickers = pd.read_csv(ruta_archivo)
print("Tickers cargados:",len(df_tickers))

Descargando constituents.csv desde GitHub...
Descarga completada.
Tickers cargados: 503


In [3]:
# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()

# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "Date added": "DateAdded"
    }, inplace=True)

# Eliminar espacios en los nombres de los sectores
df_tickers["Sector"] = df_tickers["Sector"].str.replace(" ", "")

df_tickers.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)
df_index = extraer_precios(['SPY'])
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23950 entries, 0 to 23949
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   23950 non-null  datetime64[ns]
 1   Close   23950 non-null  float64       
 2   Ticker  23950 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 561.5+ KB


In [5]:
# Calcular los retornos mensuales, varianza del activo y covarianza con el mercado para cada ticker
df_retornos = calcular_retornos(df_prices, df_index)

# Se renombran las columnas
df_retornos = df_retornos.rename(columns={
    'Fecha' : 'Date',
    'Retorno_Mensual' : 'Monthly_Return',
    'Varianza_Activo': 'Monthly_Variance',
    'Covarianza_Mercado' : 'Market_Covariance'
})
# Quitar zonas horarias de Date y convertir a object con fecha pura
df_retornos['Date'] = df_retornos['Date'].dt.tz_localize(None).dt.date

df_retornos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19432 entries, 0 to 19431
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               19432 non-null  object 
 1   Ticker             19432 non-null  object 
 2   Monthly_Return     19432 non-null  float64
 3   Monthly_Variance   19432 non-null  float64
 4   Market_Covariance  19432 non-null  float64
dtypes: float64(3), object(2)
memory usage: 759.2+ KB


In [6]:
# Extraer datos financieros de yfinance (demora 10 minutos)
df_financials = extraer_financials(tickers_list)
df_financials.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2379 entries, 0 to 2378
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Fecha                      2379 non-null   object 
 1   Total Revenue              2010 non-null   float64
 2   Gross Profit               1800 non-null   float64
 3   Operating Income           1826 non-null   float64
 4   Net Income                 2002 non-null   float64
 5   EBITDA                     1822 non-null   float64
 6   Basic Average Shares       2003 non-null   float64
 7   Cash And Cash Equivalents  2002 non-null   float64
 8   Current Debt               1601 non-null   float64
 9   Long Term Debt             1892 non-null   float64
 10  Total Debt                 1990 non-null   float64
 11  Stockholders Equity        2002 non-null   float64
 12  Total Assets               2006 non-null   float64
 13  Current Assets             1822 non-null   float

In [7]:
# Se renombran las columnas Fecha a Date
df_financials = df_financials.rename(columns={'Fecha': 'Date'})
df_prices = df_prices.rename(columns={'Fecha': 'Date'})

# Eliminar cualquier columna duplicada (para evitar errores al reejecutar la celda)
df_financials = df_financials.loc[:, ~df_financials.columns.duplicated()]
df_prices = df_prices.loc[:, ~df_prices.columns.duplicated()]

# Se convierten a datetime nativo para poder hacer cálculos
df_financials['Date'] = pd.to_datetime(df_financials['Date']).dt.tz_localize(None)
df_prices['Date'] = pd.to_datetime(df_prices['Date']).dt.tz_localize(None)

# Cálculo del primer día del mes siguiente
df_financials['Date'] = (df_financials['Date'] + pd.offsets.MonthEnd(0)) + pd.Timedelta(days=1)

# Se convierten ambos DataFrames a fecha pura sin hora
df_financials['Date'] = df_financials['Date'].dt.date
df_prices['Date'] = df_prices['Date'].dt.date

# Asegurar que los Tickers no tengan espacios en blanco invisibles
df_prices['Ticker'] = df_prices['Ticker'].astype(str).str.strip()
df_financials['Ticker'] = df_financials['Ticker'].astype(str).str.strip()

# Unir datos financieros con precios mensuales
df_merged = pd.merge(
    df_prices, 
    df_financials, 
    on=['Date', 'Ticker'],
    how='left'
)

# Ordenar cronológicamente para el Forward Fill
df_merged = df_merged.sort_values(by=['Ticker', 'Date'])

# Aplicar Forward Fill a las columnas financieras
cols_financieras = cols_resultados + cols_balance + cols_cashflow

df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

df_merged.info()


<class 'pandas.core.frame.DataFrame'>
Index: 23950 entries, 432 to 23949
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       23950 non-null  object 
 1   Close                      23950 non-null  float64
 2   Ticker                     23950 non-null  object 
 3   Total Revenue              19865 non-null  float64
 4   Gross Profit               17806 non-null  float64
 5   Operating Income           18067 non-null  float64
 6   Net Income                 19787 non-null  float64
 7   EBITDA                     18028 non-null  float64
 8   Basic Average Shares       19814 non-null  float64
 9   Cash And Cash Equivalents  19826 non-null  float64
 10  Current Debt               16611 non-null  float64
 11  Long Term Debt             18829 non-null  float64
 12  Total Debt                 19802 non-null  float64
 13  Stockholders Equity        19826 non-null  float6

In [8]:
# Eliminar las filas anteriores al primer reporte financiero disponible
columna_critica = 'EBITDA' # es necesaria para los ratios
df_merged_clean = df_merged.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los retornos calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_retornos, on=['Ticker', 'Date'], how='left')

df_merged_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18028 entries, 0 to 18027
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       18028 non-null  object 
 1   Close                      18028 non-null  float64
 2   Ticker                     18028 non-null  object 
 3   Total Revenue              18028 non-null  float64
 4   Gross Profit               17767 non-null  float64
 5   Operating Income           18028 non-null  float64
 6   Net Income                 17989 non-null  float64
 7   EBITDA                     18028 non-null  float64
 8   Basic Average Shares       17977 non-null  float64
 9   Cash And Cash Equivalents  17989 non-null  float64
 10  Current Debt               15085 non-null  float64
 11  Long Term Debt             17070 non-null  float64
 12  Total Debt                 17965 non-null  float64
 13  Stockholders Equity        17989 non-null  flo

In [9]:
df_with_metrics = calcular_metricas(df_merged_clean)

# Se crean copias de los datos financieros puros que se mantienen para el pre-procesado
# Necesarias para calcular tamaños relativos
df_with_metrics['TotalAssets'] = df_with_metrics['Total Assets']
df_with_metrics['TotalRevenue'] = df_with_metrics['Total Revenue']
df_with_metrics['TotalEquity'] = df_with_metrics['Stockholders Equity']

# Luego de calcular los features, se eliminan las columnas financieras originales
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18028 entries, 0 to 18027
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                18028 non-null  object 
 1   Close               18028 non-null  float64
 2   Ticker              18028 non-null  object 
 3   Monthly_Return      17550 non-null  float64
 4   Monthly_Variance    17550 non-null  float64
 5   Market_Covariance   17550 non-null  float64
 6   MarketCap           17977 non-null  float64
 7   EnterpriseValue     17977 non-null  float64
 8   PE_Trailing         17938 non-null  float64
 9   EnterpriseToEbitda  17977 non-null  float64
 10  PriceToBook         17938 non-null  float64
 11  operatingMargins    18028 non-null  float64
 12  profitMargins       17989 non-null  float64
 13  returnOnEquity      17950 non-null  float64
 14  ReturnOnAssets      17989 non-null  float64
 15  debtToEquity        17989 non-null  float64
 16  curr

In [10]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18028 entries, 0 to 18027
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                18028 non-null  object 
 1   Close               18028 non-null  float64
 2   Ticker              18028 non-null  object 
 3   Monthly_Return      17550 non-null  float64
 4   Monthly_Variance    17550 non-null  float64
 5   Market_Covariance   17550 non-null  float64
 6   MarketCap           17977 non-null  float64
 7   EnterpriseValue     17977 non-null  float64
 8   PE_Trailing         17938 non-null  float64
 9   EnterpriseToEbitda  17977 non-null  float64
 10  PriceToBook         17938 non-null  float64
 11  operatingMargins    18028 non-null  float64
 12  profitMargins       17989 non-null  float64
 13  returnOnEquity      17950 non-null  float64
 14  ReturnOnAssets      17989 non-null  float64
 15  debtToEquity        17989 non-null  float64
 16  curr

In [ ]:
# Guardar datos extraidos en fichero raw_data
df_final.to_parquet(f"data/raw_data.parquet")